<a href="https://colab.research.google.com/github/goumze/Simplilearn_Agentic_AI/blob/feature%2Fcollab/Copy_of_Simple_Agent_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -r requirements.txt

  Using cached langchain_core-0.2.38-py3-none-any.whl.metadata (6.2 kB)
Using cached langchain_core-0.2.38-py3-none-any.whl (396 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.2.43
    Uninstalling langchain-core-0.2.43:
      Successfully uninstalled langchain-core-0.2.43
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 0.2.2 requires langchain-core!=0.3.0,!=0.3.1,!=0.3.10,!=0.3.11,!=0.3.12,!=0.3.13,!=0.3.14,!=0.3.2,!=0.3.3,!=0.3.4,!=0.3.5,!=0.3.6,!=0.3.7,!=0.3.8,!=0.3.9,<0.4.0,>=0.2.43, but you have langchain-core 0.2.38 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.2.38 which is incompatible.


In [ ]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import requests
import json
load_dotenv(override=True)

False

In [ ]:
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')
openAI = OpenAI(api_key=openai_api_key)

In [ ]:
# --- Tool 1: Add two numbers ---
def add(a:int, b:int) -> str:
  return f"{a}+{b} = {a+b}"

In [ ]:
# --- Tool 2: Get real weather using wttr.in (no API key needed) ---
def get_weather(city: str) -> str:
    try:
        # Step 1: Convert city name → latitude/longitude
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=5
        ).json()

        if not geo.get("results"):
            return f"City '{city}' not found."

        result = geo["results"][0]
        lat, lon, name = result["latitude"], result["longitude"], result["name"]

        # Step 2: Fetch current weather using the coordinates
        weather = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lon,
                "current": "temperature_2m,relative_humidity_2m,windspeed_10m,weathercode"
            },
            timeout=5
        ).json()

        c = weather["current"]
        return (
            f"{name}: {c['temperature_2m']}°C, "
            f"Humidity {c['relative_humidity_2m']}%, "
            f"Wind {c['windspeed_10m']} km/h"
        )
    except Exception as e:
        return f"Could not fetch weather for {city}: {e}"


In [ ]:
# --- JSON schemas: describe the tools to the AI ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two numbers together",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer", "description": "First number"},
                    "b": {"type": "integer", "description": "Second number"}
                },
                "required": ["a", "b"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Name of the city"}
                },
                "required": ["city"],
                "additionalProperties": False
            }
        }
    }
]

In [ ]:
def handle_tool_calls(tool_calls):
    """Run each tool the AI requested and return the results."""
    results = []
    for tc in tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments)
        print(f"  → AI called: {name}({args})")

        fn = globals().get(name)
        output = fn(**args) if fn else "Tool not found"
        print(f"  ← Result: {output}\n")

        results.append({"role": "tool", "content": output, "tool_call_id": tc.id})
    return results

In [ ]:
def agent_loop(messages):
    """Keep calling the AI until it stops using tools and gives a final answer."""
    while True:
        response = openAI.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )
        choice = response.choices[0]

        if choice.finish_reason == "tool_calls":
            print("--- AI is calling tools ---")
            tool_results = handle_tool_calls(choice.message.tool_calls)
            messages.append(choice.message)   # AI's tool request
            messages.extend(tool_results)     # Our tool results
        else:
            print("--- Final Answer ---")
            print(choice.message.content)
            break


In [ ]:
messages = [
    {"role": "user", "content": "What is 15 + 27? Also, what's the weather in Paris?"}
]

agent_loop(messages)


--- AI is calling tools ---
  → AI called: add({'a': 15, 'b': 27})
  ← Result: 15+27 = 42

  → AI called: get_weather({'city': 'Paris'})
  ← Result: Paris: 13.0°C, Humidity 50%, Wind 11.6 km/h

--- Final Answer ---
The result of \( 15 + 27 \) is \( 42 \). 

As for the weather in Paris, it is currently \( 13.0°C \) with a humidity of \( 50\% \) and wind speed of \( 11.6 \, \text{km/h} \).


---

# 🔍 Production Scenario: E-Commerce Order Fraud Detection

## How this works (Two-Phase Agentic Loop)

```
Phase 1 — Agent Loop                     Phase 2 — Evaluator
─────────────────────────────────────    ──────────────────────────────
User gives an order ID                   Agent's full assessment is passed
     ↓                                   to a second LLM acting as a
Agent calls get_order()                  senior fraud analyst
     ↓                                        ↓
Agent calls get_customer_profile()       Evaluator reasons independently
     ↓                                        ↓
Agent calls calculate_risk_score()       Returns final verdict:
     ↓                                   APPROVE / FLAG / BLOCK
Agent writes a detailed assessment
```

> 💡 **Why two phases?** The agent is fast but may miss nuance. The evaluator adds a second layer of reasoning — like a junior analyst handing off to a senior reviewer before a decision is made.


In [ ]:
# ── Fake production data ──────────────────────────────────────────────────────

ORDERS = {
    "ORD-1042": {"customer_id": "CUS-88", "amount": 4200, "items": 3,  "country": "Nigeria",   "payment": "prepaid card"},
    "ORD-1043": {"customer_id": "CUS-21", "amount":   89, "items": 1,  "country": "USA",        "payment": "credit card"},
    "ORD-1044": {"customer_id": "CUS-55", "amount": 1850, "items": 5,  "country": "Romania",   "payment": "prepaid card"},
    "ORD-1045": {"customer_id": "CUS-10", "amount":  320, "items": 2,  "country": "UK",         "payment": "debit card"},
}

CUSTOMERS = {
    "CUS-88": {"name": "Alex Turner",   "total_orders":  1,  "returns": 0, "account_age_days":   2},
    "CUS-21": {"name": "Sarah Chen",    "total_orders": 47,  "returns": 2, "account_age_days": 720},
    "CUS-55": {"name": "Radu Ionescu",  "total_orders":  3,  "returns": 1, "account_age_days":  15},
    "CUS-10": {"name": "James Wright",  "total_orders": 130, "returns": 5, "account_age_days": 1400},
}



In [ ]:
# ── Tool functions ─────────────────────────────────────────────────────────────

def get_order(order_id: str) -> str:
    """Return order details for a given order ID."""
    order = ORDERS.get(order_id)
    return json.dumps(order) if order else f"Order '{order_id}' not found."

def get_customer_profile(customer_id: str) -> str:
    """Return customer profile for a given customer ID."""
    customer = CUSTOMERS.get(customer_id)
    return json.dumps(customer) if customer else f"Customer '{customer_id}' not found."

def calculate_risk_score(high_value: bool, new_account: bool, unusual_location: bool, prepaid_card: bool) -> str:
    """Compute a fraud risk score from boolean risk factors."""
    score = (high_value * 30) + (new_account * 25) + (unusual_location * 25) + (prepaid_card * 20)
    level = "HIGH" if score >= 50 else "MEDIUM" if score >= 25 else "LOW"
    return f"Risk Score: {score}/100 — {level} RISK"

In [ ]:
# ── Tool JSON schemas ──────────────────────────────────────────────────────────

fraud_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order",
            "description": "Retrieve order details (amount, country, payment method) by order ID",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string", "description": "The order ID to look up"}},
                "required": ["order_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_customer_profile",
            "description": "Retrieve a customer's history (order count, returns, account age)",
            "parameters": {
                "type": "object",
                "properties": {"customer_id": {"type": "string", "description": "The customer ID to look up"}},
                "required": ["customer_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_risk_score",
            "description": "Compute a fraud risk score based on four boolean risk factors",
            "parameters": {
                "type": "object",
                "properties": {
                    "high_value":         {"type": "boolean", "description": "True if order amount > 1000"},
                    "new_account":        {"type": "boolean", "description": "True if account is < 30 days old"},
                    "unusual_location":   {"type": "boolean", "description": "True if country is not USA, UK, Canada, or Australia"},
                    "prepaid_card":       {"type": "boolean", "description": "True if payment method contains 'prepaid'"}
                },
                "required": ["high_value", "new_account", "unusual_location", "prepaid_card"],
                "additionalProperties": False
            }
        }
    }
]

In [ ]:
def fraud_handle_tool_calls(tool_calls):
    """Execute fraud detection tools and return results."""
    results = []
    for tc in tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments)
        print(f"  → Agent called: {name}({args})")
        fn = globals().get(name)
        output = fn(**args) if fn else "Tool not found"
        print(f"  ← Result: {output}\n")
        results.append({"role": "tool", "content": str(output), "tool_call_id": tc.id})
    return results

In [ ]:
# ── Phase 1: Agent Loop ────────────────────────────────────────────────────────

def fraud_agent_loop(messages) -> str:
    """
    Run the agent loop for fraud detection.
    Returns the agent's final assessment text (used by the evaluator in Phase 2).
    """
    while True:
        response = openAI.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=fraud_tools
        )
        choice = response.choices[0]

        if choice.finish_reason == "tool_calls":
            print("  [Agent is calling tools...]")
            tool_results = fraud_handle_tool_calls(choice.message.tool_calls)
            messages.append(choice.message)
            messages.extend(tool_results)
        else:
            assessment = choice.message.content
            print("\n  [Agent Assessment Complete]")
            print(f"  {assessment}\n")
            return assessment   # ← Return to Phase 2

In [ ]:
# ── Phase 2: Evaluator ────────────────────────────────────────────────────────

def evaluate_assessment(agent_assessment: str, order_id: str) -> str:
    """
    A second LLM call — acts as a senior fraud analyst reviewing the agent's work.
    Returns a final structured verdict: APPROVE, FLAG, or BLOCK.
    """
    eval_messages = [
        {
            "role": "system",
            "content": (
                "You are a senior fraud analyst at an e-commerce company.\n"
                "A junior AI agent has assessed an order for fraud risk. "
                "Your job is to review its reasoning, then issue a final decision.\n\n"
                "Respond strictly in this format:\n"
                "DECISION: <APPROVE | FLAG FOR REVIEW | BLOCK>\n"
                "REASON: <one concise sentence explaining your decision>"
            )
        },
        {
            "role": "user",
            "content": f"Order ID: {order_id}\n\nAgent's Assessment:\n{agent_assessment}"
        }
    ]

    response = openAI.chat.completions.create(
        model="gpt-4o-mini",
        messages=eval_messages
    )
    return response.choices[0].message.content

In [ ]:
ORDER_TO_CHECK = "ORD-1043"   # ← Try: ORD-1042, ORD-1043, ORD-1044, ORD-1045

print(f"{'═'*55}")
print(f"  FRAUD DETECTION SYSTEM  |  Order: {ORDER_TO_CHECK}")
print(f"{'═'*55}\n")

fraud_messages = [
    {
        "role": "system",
        "content": (
            "You are a fraud detection agent for an e-commerce platform.\n"
            "For every order you receive:\n"
            "1. Call get_order() to retrieve order details\n"
            "2. Call get_customer_profile() using the customer_id from the order\n"
            "3. Call calculate_risk_score() with:\n"
            "   - high_value: True if amount > 1000\n"
            "   - new_account: True if account_age_days < 30\n"
            "   - unusual_location: True if country is not USA, UK, Canada, or Australia\n"
            "   - prepaid_card: True if payment method contains 'prepaid'\n"
            "4. Write a clear, structured assessment summarising what you found and why the score is justified."
        )
    },
    {
        "role": "user",
        "content": f"Assess order {ORDER_TO_CHECK} for potential fraud."
    }
]

# ── Phase 1: Agent gathers data and writes assessment ─────────────────────────
print("PHASE 1 — Agent Investigation\n")
agent_assessment = fraud_agent_loop(fraud_messages)

# ── Phase 2: Evaluator reviews and issues final verdict ───────────────────────
print("PHASE 2 — Evaluator Verdict\n")
verdict = evaluate_assessment(agent_assessment, ORDER_TO_CHECK)
print(f"  {verdict}")
print(f"\n{'═'*55}")


═══════════════════════════════════════════════════════
  FRAUD DETECTION SYSTEM  |  Order: ORD-1043
═══════════════════════════════════════════════════════

PHASE 1 — Agent Investigation

  [Agent is calling tools...]
  → Agent called: get_order({'order_id': 'ORD-1043'})
  ← Result: {"customer_id": "CUS-21", "amount": 89, "items": 1, "country": "USA", "payment": "credit card"}

  [Agent is calling tools...]
  → Agent called: get_customer_profile({'customer_id': 'CUS-21'})
  ← Result: {"name": "Sarah Chen", "total_orders": 47, "returns": 2, "account_age_days": 720}

  [Agent is calling tools...]
  → Agent called: calculate_risk_score({'high_value': False, 'new_account': False, 'unusual_location': False, 'prepaid_card': False})
  ← Result: Risk Score: 0/100 — LOW RISK


  [Agent Assessment Complete]
  ### Fraud Assessment for Order ORD-1043

**Order Details:**
- **Customer ID:** CUS-21
- **Order Amount:** $89
- **Country:** USA
- **Payment Method:** Credit Card

**Customer Profile:**
- 

Implementing RAG context to the above use case

In [ ]:
pip install chromadb

In [ ]:
import chromadb

# ── Seed data: past fraud cases with known outcomes ───────────────────────────
PAST_CASES = [
    {"id": "HIST-001", "doc": "High value $5200, account 1 day old, prepaid card, Nigeria, no order history.",       "meta": {"outcome": "BLOCKED"}},
    {"id": "HIST-002", "doc": "Low value $75, established customer 3 years, 50 orders, credit card, USA.",           "meta": {"outcome": "APPROVED"}},
    {"id": "HIST-003", "doc": "High value $1500, account 10 days old, prepaid card, Romania, 2 prior orders.",       "meta": {"outcome": "FLAGGED"}},
    {"id": "HIST-004", "doc": "High value $3200, prepaid card, Vietnam, account 5 days old, no history.",            "meta": {"outcome": "BLOCKED"}},
    {"id": "HIST-005", "doc": "Order $250, loyal customer 4 years, 120 orders, debit card, UK.",                     "meta": {"outcome": "APPROVED"}},
    {"id": "HIST-006", "doc": "Order $1900, account 18 days old, prepaid card, Eastern Europe, 3 orders.",           "meta": {"outcome": "FLAGGED"}},
    {"id": "HIST-007", "doc": "High value $4800, account 3 days, prepaid card, West Africa, zero order history.",    "meta": {"outcome": "BLOCKED"}},
    {"id": "HIST-008", "doc": "Order $110, returning customer 2 years, 30 orders, credit card, Canada.",             "meta": {"outcome": "APPROVED"}},
    {"id": "HIST-009", "doc": "High value $2600, new account 7 days, prepaid card, Russia.",                         "meta": {"outcome": "BLOCKED"}},
    {"id": "HIST-010", "doc": "Order $430, account 6 months, 8 orders, debit card, USA, low returns.",               "meta": {"outcome": "APPROVED"}},
]

# ── Create in-memory ChromaDB collection and load historical cases ─────────────
chroma_client = chromadb.Client()
fraud_collection = chroma_client.create_collection(name="fraud_history")
fraud_collection.add(
    documents=[c["doc"]  for c in PAST_CASES],
    metadatas= [c["meta"] for c in PAST_CASES],
    ids=       [c["id"]   for c in PAST_CASES],
)
print(f"✅ ChromaDB in-memory store loaded with {len(PAST_CASES)} historical fraud cases.")

# ── RAG tool: semantic search over past cases ──────────────────────────────────
def retrieve_fraud_cases(query: str) -> str:
    """Retrieve the 3 most similar past fraud cases from ChromaDB."""
    results = fraud_collection.query(query_texts=[query], n_results=3)
    lines = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        lines.append(f"  [{meta['outcome']}] {doc}")
    return "\n".join(lines)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ ChromaDB in-memory store loaded with 10 historical fraud cases.


In [ ]:
# ── Fake production data ──────────────────────────────────────────────────────

ORDERS = {
    "ORD-1042": {"customer_id": "CUS-88", "amount": 4200, "items": 3,  "country": "Nigeria",  "payment": "prepaid card"},
    "ORD-1043": {"customer_id": "CUS-21", "amount":   89, "items": 1,  "country": "USA",       "payment": "credit card"},
    "ORD-1044": {"customer_id": "CUS-55", "amount": 1850, "items": 5,  "country": "Romania",  "payment": "prepaid card"},
    "ORD-1045": {"customer_id": "CUS-10", "amount":  320, "items": 2,  "country": "UK",        "payment": "debit card"},
}

CUSTOMERS = {
    "CUS-88": {"name": "Alex Turner",  "total_orders":   1, "returns": 0, "account_age_days":    2},
    "CUS-21": {"name": "Sarah Chen",   "total_orders":  47, "returns": 2, "account_age_days":  720},
    "CUS-55": {"name": "Radu Ionescu", "total_orders":   3, "returns": 1, "account_age_days":   15},
    "CUS-10": {"name": "James Wright", "total_orders": 130, "returns": 5, "account_age_days": 1400},
}

# ── Tool functions ─────────────────────────────────────────────────────────────

def get_order(order_id: str) -> str:
    order = ORDERS.get(order_id)
    return json.dumps(order) if order else f"Order '{order_id}' not found."

def get_customer_profile(customer_id: str) -> str:
    customer = CUSTOMERS.get(customer_id)
    return json.dumps(customer) if customer else f"Customer '{customer_id}' not found."

def calculate_risk_score(high_value: bool, new_account: bool, unusual_location: bool, prepaid_card: bool) -> str:
    score = (high_value * 30) + (new_account * 25) + (unusual_location * 25) + (prepaid_card * 20)
    level = "HIGH" if score >= 50 else "MEDIUM" if score >= 25 else "LOW"
    return f"Risk Score: {score}/100 — {level} RISK"

# ── Tool JSON schemas ──────────────────────────────────────────────────────────

fraud_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order",
            "description": "Retrieve order details (amount, country, payment method) by order ID",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string", "description": "The order ID to look up"}},
                "required": ["order_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_customer_profile",
            "description": "Retrieve a customer's history (order count, returns, account age)",
            "parameters": {
                "type": "object",
                "properties": {"customer_id": {"type": "string", "description": "The customer ID to look up"}},
                "required": ["customer_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "retrieve_fraud_cases",
            "description": "Search ChromaDB for similar historical fraud cases and their outcomes (APPROVED / FLAGGED / BLOCKED) to guide the current assessment",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Natural language description of the current order's risk factors, e.g. 'high value, new account, prepaid card, Nigeria'"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_risk_score",
            "description": "Compute a fraud risk score based on four boolean risk factors",
            "parameters": {
                "type": "object",
                "properties": {
                    "high_value":       {"type": "boolean", "description": "True if order amount > 1000"},
                    "new_account":      {"type": "boolean", "description": "True if account is < 30 days old"},
                    "unusual_location": {"type": "boolean", "description": "True if country is not USA, UK, Canada, or Australia"},
                    "prepaid_card":     {"type": "boolean", "description": "True if payment method contains 'prepaid'"}
                },
                "required": ["high_value", "new_account", "unusual_location", "prepaid_card"],
                "additionalProperties": False
            }
        }
    }
]


In [ ]:
def fraud_handle_tool_calls(tool_calls):
    """Execute fraud detection tools and return results."""
    results = []
    for tc in tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments)
        print(f"  → Agent called: {name}({args})")
        fn = globals().get(name)
        output = fn(**args) if fn else "Tool not found"
        print(f"  ← Result: {output}\n")
        results.append({"role": "tool", "content": str(output), "tool_call_id": tc.id})
    return results


# ── Phase 1: Agent Loop ────────────────────────────────────────────────────────

def fraud_agent_loop(messages) -> str:
    """
    Agent loop: gathers order data, retrieves similar past cases from ChromaDB,
    computes risk score, then writes a structured assessment.
    Returns the assessment text for the evaluator in Phase 2.
    """
    while True:
        response = openAI.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=fraud_tools
        )
        choice = response.choices[0]

        if choice.finish_reason == "tool_calls":
            print("  [Agent is calling tools...]")
            tool_results = fraud_handle_tool_calls(choice.message.tool_calls)
            messages.append(choice.message)
            messages.extend(tool_results)
        else:
            assessment = choice.message.content
            print("\n  [Agent Assessment Complete]")
            print(f"  {assessment}\n")
            return assessment   # ← passed to Phase 2


# ── Phase 2: Evaluator with RAG ───────────────────────────────────────────────

def evaluate_assessment(agent_assessment: str, order_id: str) -> str:
    """
    Senior analyst evaluation with RAG:
    - Independently queries ChromaDB for similar past cases
    - Reviews the agent's assessment alongside retrieved history
    - Issues a final structured verdict: APPROVE / FLAG FOR REVIEW / BLOCK
    """
    # RAG: pull 3 semantically similar past cases from ChromaDB
    rag_context = retrieve_fraud_cases(agent_assessment[:300])
    print(f"  [Evaluator retrieved 3 similar historical cases from ChromaDB]\n")
    for line in rag_context.splitlines():
        print(f"  {line}")
    print()

    eval_messages = [
        {
            "role": "system",
            "content": (
                "You are a senior fraud analyst at an e-commerce company.\n"
                "You are given:\n"
                "  1. A junior AI agent's assessment of a suspicious order\n"
                "  2. Similar historical cases retrieved from the company's fraud database (ChromaDB)\n\n"
                "Use BOTH sources of information to issue your final decision.\n\n"
                "Respond strictly in this format:\n"
                "DECISION: <APPROVE | FLAG FOR REVIEW | BLOCK>\n"
                "REASON: <one concise sentence citing specific evidence from the assessment and/or historical cases>"
            )
        },
        {
            "role": "user",
            "content": (
                f"Order ID: {order_id}\n\n"
                f"--- Similar Historical Cases (ChromaDB RAG) ---\n{rag_context}\n\n"
                f"--- Agent's Assessment ---\n{agent_assessment}"
            )
        }
    ]

    response = openAI.chat.completions.create(
        model="gpt-4o-mini",
        messages=eval_messages
    )
    return response.choices[0].message.content


In [ ]:
ORDER_TO_CHECK = "ORD-1042"   # ← Try: ORD-1042, ORD-1043, ORD-1044, ORD-1045

print(f"{'═'*60}")
print(f"  FRAUD DETECTION SYSTEM (RAG-enhanced)  |  Order: {ORDER_TO_CHECK}")
print(f"{'═'*60}\n")

fraud_messages = [
    {
        "role": "system",
        "content": (
            "You are a fraud detection agent for an e-commerce platform.\n"
            "For every order you receive, follow these steps in order:\n"
            "1. Call get_order() to retrieve order details\n"
            "2. Call get_customer_profile() using the customer_id from the order\n"
            "3. Call retrieve_fraud_cases() with a short description of the key risk factors found so far\n"
            "   — this returns similar past cases from the company's historical fraud database\n"
            "4. Call calculate_risk_score() with the appropriate boolean flags\n"
            "5. Write a structured assessment covering:\n"
            "   - Order & customer summary\n"
            "   - Risk score and what drove it\n"
            "   - What the similar historical cases suggest\n"
            "   - Your overall conclusion"
        )
    },
    {
        "role": "user",
        "content": f"Assess order {ORDER_TO_CHECK} for potential fraud."
    }
]

# ── Phase 1: Agent investigates using tools + ChromaDB RAG ───────────────────
print("PHASE 1 — Agent Investigation (with RAG)\n")
agent_assessment = fraud_agent_loop(fraud_messages)

# ── Phase 2: Evaluator issues verdict using agent output + ChromaDB RAG ──────
print("PHASE 2 — Evaluator Verdict (with RAG)\n")
verdict = evaluate_assessment(agent_assessment, ORDER_TO_CHECK)
print(f"  {verdict}")
print(f"\n{'═'*60}")


════════════════════════════════════════════════════════════
  FRAUD DETECTION SYSTEM (RAG-enhanced)  |  Order: ORD-1042
════════════════════════════════════════════════════════════

PHASE 1 — Agent Investigation (with RAG)

  [Agent is calling tools...]
  → Agent called: get_order({'order_id': 'ORD-1042'})
  ← Result: {"customer_id": "CUS-88", "amount": 4200, "items": 3, "country": "Nigeria", "payment": "prepaid card"}

  [Agent is calling tools...]
  → Agent called: get_customer_profile({'customer_id': 'CUS-88'})
  ← Result: {"name": "Alex Turner", "total_orders": 1, "returns": 0, "account_age_days": 2}



ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [Agent is calling tools...]
  → Agent called: retrieve_fraud_cases({'query': 'high value, new account, prepaid card, Nigeria'})
  ← Result:   [BLOCKED] High value $5200, account 1 day old, prepaid card, Nigeria, no order history.
  [BLOCKED] High value $4800, account 3 days, prepaid card, West Africa, zero order history.
  [BLOCKED] High value $2600, new account 7 days, prepaid card, Russia.

  [Agent is calling tools...]
  → Agent called: calculate_risk_score({'high_value': True, 'new_account': True, 'unusual_location': True, 'prepaid_card': True})
  ← Result: Risk Score: 100/100 — HIGH RISK


  [Agent Assessment Complete]
  ### Structured Assessment of Order ORD-1042

#### Order & Customer Summary
- **Order ID**: ORD-1042
- **Customer Name**: Alex Turner
- **Order Amount**: $4200
- **Items in Order**: 3
- **Country**: Nigeria
- **Payment Method**: Prepaid Card
- **Customer Profile**:
  - Total Orders: 1
  - Returns: 0
  - Account Age: 2 days

#### Risk Score and What Drove It
The c

---

# 🦜 LangChain-Powered Upgrade: Fraud Detection

## Raw OpenAI API vs LangChain

| | Raw OpenAI (above) | LangChain (below) |
|---|---|---|
| **Tool definition** | Manual JSON schemas | `@tool` decorator — just a docstring |
| **Agent loop** | `while True` written by hand | `AgentExecutor` handles it |
| **RAG** | Direct ChromaDB calls | `Chroma` retriever abstraction |
| **Evaluator** | Manual message list | LCEL chain: `prompt \| llm \| parser` |
| **Prompt management** | Inline strings | `ChatPromptTemplate` |

> The logic is identical — LangChain just removes all the boilerplate.


In [ ]:
# Install if needed:
!pip install langchain langchain-openai langchain-chroma langchain-community

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import AgentExecutor, create_openai_tools_agent

# LangChain LLM — same model, but wrapped in LangChain's ChatOpenAI
lc_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=openai_api_key)
print("✅ LangChain LLM ready.")

✅ LangChain LLM ready.


In [ ]:
# ── LangChain Chroma vector store with real OpenAI embeddings ─────────────────
# Uses OpenAI's text-embedding-3-small to embed documents — proper semantic search
lc_fraud_store = Chroma.from_texts(
    texts=    [c["doc"]  for c in PAST_CASES],
    metadatas=[c["meta"] for c in PAST_CASES],
    embedding=OpenAIEmbeddings(model="text-embedding-3-small",openai_api_key=openai_api_key),
    collection_name="lc_fraud_history",
)
retriever = lc_fraud_store.as_retriever(search_kwargs={"k": 3})
print(f"✅ LangChain Chroma store ready — {lc_fraud_store._collection.count()} embedded documents.\n")

# ── Tools via @tool decorator — no JSON schemas needed ────────────────────────
# LangChain reads the function signature + docstring to build the schema automatically

@tool
def lc_get_order(order_id: str) -> str:
    """Retrieve order details including amount, country, and payment method by order ID."""
    order = ORDERS.get(order_id)
    return json.dumps(order) if order else f"Order '{order_id}' not found."

@tool
def lc_get_customer_profile(customer_id: str) -> str:
    """Retrieve a customer's full history: total orders, returns, and account age in days."""
    customer = CUSTOMERS.get(customer_id)
    return json.dumps(customer) if customer else f"Customer '{customer_id}' not found."

@tool
def lc_retrieve_fraud_cases(query: str) -> str:
    """Search the fraud history database (ChromaDB) for the 3 most similar past cases.
    Returns each case with its outcome label: APPROVED, FLAGGED, or BLOCKED.
    Use this after collecting order and customer details to find relevant precedents."""
    docs = retriever.invoke(query)
    return "\n".join([f"[{d.metadata['outcome']}] {d.page_content}" for d in docs])

@tool
def lc_calculate_risk_score(high_value: bool, new_account: bool, unusual_location: bool, prepaid_card: bool) -> str:
    """Compute a fraud risk score (0-100) from four boolean risk factors.
    high_value=True if order amount > $1000.
    new_account=True if account is less than 30 days old.
    unusual_location=True if country is not USA, UK, Canada, or Australia.
    prepaid_card=True if payment method contains the word 'prepaid'."""
    score = (high_value * 30) + (new_account * 25) + (unusual_location * 25) + (prepaid_card * 20)
    level = "HIGH" if score >= 50 else "MEDIUM" if score >= 25 else "LOW"
    return f"Risk Score: {score}/100 — {level} RISK"

lc_tools = [lc_get_order, lc_get_customer_profile, lc_retrieve_fraud_cases, lc_calculate_risk_score]
print(f"✅ LangChain @tools registered: {[t.name for t in lc_tools]}")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ LangChain Chroma store ready — 20 embedded documents.

✅ LangChain @tools registered: ['lc_get_order', 'lc_get_customer_profile', 'lc_retrieve_fraud_cases', 'lc_calculate_risk_score']


In [ ]:
# ── Phase 1: LangChain Agent with AgentExecutor ───────────────────────────────
# create_openai_tools_agent builds the agent; AgentExecutor runs the loop
agent_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a fraud detection agent for an e-commerce platform.\n"
     "For every order, follow these steps in order:\n"
     "1. Call lc_get_order to retrieve order details\n"
     "2. Call lc_get_customer_profile using the customer_id from the order\n"
     "3. Call lc_retrieve_fraud_cases with a short description of the risk factors found\n"
     "4. Call lc_calculate_risk_score with the appropriate boolean flags\n"
     "5. Write a structured assessment: order summary, risk score, historical case insights, conclusion."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),  # required: holds tool call history
])

lc_agent = create_openai_tools_agent(lc_llm, lc_tools, agent_prompt)
# verbose=True makes AgentExecutor print each tool call — great for learning
lc_agent_executor = AgentExecutor(agent=lc_agent, tools=lc_tools, verbose=True)

# ── Phase 2: Evaluator as an LCEL chain ───────────────────────────────────────
# LCEL syntax: prompt | llm | output_parser  — each step's output feeds the next
evaluator_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior fraud analyst.\n"
     "You are given a junior AI agent's assessment and similar historical cases from the fraud database.\n"
     "Use BOTH to issue your final decision.\n\n"
     "Respond strictly in this format:\n"
     "DECISION: <APPROVE | FLAG FOR REVIEW | BLOCK>\n"
     "REASON: <one concise sentence citing specific evidence>"),
    ("human",
     "Order ID: {order_id}\n\n"
     "--- Similar Historical Cases (LangChain ChromaDB RAG) ---\n{rag_context}\n\n"
     "--- Agent's Assessment ---\n{assessment}"),
])

# The pipe operator | chains: prompt → LLM → string parser
evaluator_chain = evaluator_prompt | lc_llm | StrOutputParser()

print("✅ LangChain AgentExecutor and LCEL evaluator chain ready.")


✅ LangChain AgentExecutor and LCEL evaluator chain ready.


In [ ]:
LC_ORDER = "ORD-1042"   # ← Try: ORD-1042, ORD-1043, ORD-1044, ORD-1045

print(f"{'═'*60}")
print(f"  LANGCHAIN FRAUD SYSTEM  |  Order: {LC_ORDER}")
print(f"{'═'*60}\n")

# ── Phase 1: AgentExecutor runs the tool-calling loop automatically ───────────
print("PHASE 1 — LangChain AgentExecutor (verbose=True shows each step)\n")
result = lc_agent_executor.invoke({"input": f"Assess order {LC_ORDER} for potential fraud."})
lc_assessment = result["output"]

# ── Phase 2: LCEL evaluator chain — retrieves RAG context then pipes through ──
print("\nPHASE 2 — LCEL Evaluator Chain\n")

# Retriever fetches semantically similar cases using OpenAI embeddings
rag_docs = retriever.invoke(lc_assessment[:300])
rag_context = "\n".join([f"  [{d.metadata['outcome']}] {d.page_content}" for d in rag_docs])
print("  [Retrieved similar historical cases from LangChain Chroma]\n")
print(rag_context + "\n")

# LCEL chain: evaluator_prompt | lc_llm | StrOutputParser()
verdict = evaluator_chain.invoke({
    "order_id":   LC_ORDER,
    "rag_context": rag_context,
    "assessment":  lc_assessment,
})
print(f"  {verdict}")
print(f"\n{'═'*60}")


════════════════════════════════════════════════════════════
  LANGCHAIN FRAUD SYSTEM  |  Order: ORD-1042
════════════════════════════════════════════════════════════

PHASE 1 — LangChain AgentExecutor (verbose=True shows each step)



> Entering new AgentExecutor chain...

Invoking: `lc_get_order` with `{'order_id': 'ORD-1042'}`


{"customer_id": "CUS-88", "amount": 4200, "items": 3, "country": "Nigeria", "payment": "prepaid card"}
Invoking: `lc_get_customer_profile` with `{'customer_id': 'CUS-88'}`


{"name": "Alex Turner", "total_orders": 1, "returns": 0, "account_age_days": 2}
Invoking: `lc_retrieve_fraud_cases` with `{'query': 'High order amount, new account, unusual location, prepaid card payment.'}`




ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[FLAGGED] High value $1500, account 10 days old, prepaid card, Romania, 2 prior orders.
[FLAGGED] High value $1500, account 10 days old, prepaid card, Romania, 2 prior orders.
[FLAGGED] Order $1900, account 18 days old, prepaid card, Eastern Europe, 3 orders.
Invoking: `lc_calculate_risk_score` with `{'high_value': True, 'new_account': True, 'unusual_location': True, 'prepaid_card': True}`


Risk Score: 100/100 — HIGH RISK### Order Assessment Summary

**Order Summary:**
- **Order ID:** ORD-1042
- **Customer Name:** Alex Turner
- **Order Amount:** $4200
- **Items Ordered:** 3
- **Country:** Nigeria
- **Payment Method:** Prepaid Card
- **Customer Profile:** 
  - Total Orders: 1
  - Returns: 0
  - Account Age: 2 days

**Risk Score:** 100/100 — HIGH RISK

**Historical Case Insights:**
- **Case 1:** High value $1500, account 10 days old, prepaid card, Romania, 2 prior orders. **Outcome:** FLAGGED
- **Case 2:** High value $1500, account 10 days old, prepaid card, Romania, 2 prior orders. **O

In [ ]:
---

# 🤖 OpenAI Agents SDK: Fraud Detection (True Agentic)

## Raw OpenAI API vs LangChain vs Agents SDK

| | Raw OpenAI | LangChain | **Agents SDK** |
|---|---|---|---|
| **Tool definition** | Manual JSON schemas | `@tool` decorator | `@function_tool` decorator |
| **Agent loop** | `while True` by hand | `AgentExecutor` | `Runner.run_sync()` |
| **Multi-agent** | Not supported | `RunnableSequence` | `handoff()` — native |
| **RAG** | Direct ChromaDB calls | `Chroma` retriever | `@function_tool` wraps it |
| **Evaluator** | Separate manual call | LCEL chain `prompt \| llm` | Second `Agent` + auto-handoff |

## How the Agents SDK flow works

```
Investigator Agent                        Evaluator Agent
──────────────────────────────────────    ──────────────────────────────
Runner.run_sync() starts the agent        Agent receives investigation
     ↓                                    via handoff() automatically
Calls get_order()                              ↓
     ↓                                    Independently reasons
Calls get_customer_profile()                   ↓
     ↓                                    Issues final verdict:
Calls retrieve_fraud_cases() ← RAG        APPROVE / FLAG / BLOCK
     ↓
Calls calculate_risk_score()
     ↓
Writes assessment → handoff(evaluator)  ← agent decides to hand off
```

> 💡 **Key shift**: The agent autonomously decides WHEN to hand off to the evaluator.
> You don't call `evaluate_assessment()` manually — the agent calls `handoff()` itself.


In [ ]:
# Install if needed:
# !pip install openai-agents

from agents import Agent, Runner, function_tool, handoff
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
import asyncio

print("✅ OpenAI Agents SDK imported successfully.")


In [ ]:
# ── Tools via @function_tool — SDK reads the docstring to build JSON schema ───
# No manual JSON schema dicts needed — identical logic to cells above

@function_tool
def sdk_get_order(order_id: str) -> str:
    """Retrieve order details (amount, country, payment method) by order ID."""
    order = ORDERS.get(order_id)
    return json.dumps(order) if order else f"Order '{order_id}' not found."

@function_tool
def sdk_get_customer_profile(customer_id: str) -> str:
    """Retrieve customer history: total orders, returns, account age in days."""
    customer = CUSTOMERS.get(customer_id)
    return json.dumps(customer) if customer else f"Customer '{customer_id}' not found."

@function_tool
def sdk_retrieve_fraud_cases(query: str) -> str:
    """Search ChromaDB for the 3 most similar historical fraud cases and their outcomes.
    Pass a short description of the current order's risk factors as the query.
    Returns each case labelled APPROVED, FLAGGED, or BLOCKED."""
    results = fraud_collection.query(query_texts=[query], n_results=3)
    lines = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        lines.append(f"[{meta['outcome']}] {doc}")
    return "\n".join(lines)

@function_tool
def sdk_calculate_risk_score(
    high_value: bool,
    new_account: bool,
    unusual_location: bool,
    prepaid_card: bool,
) -> str:
    """Compute a fraud risk score (0–100) from four boolean risk factors.
    high_value: True if order amount > $1000.
    new_account: True if account is less than 30 days old.
    unusual_location: True if country is not USA, UK, Canada, or Australia.
    prepaid_card: True if payment method contains the word 'prepaid'."""
    score = (high_value * 30) + (new_account * 25) + (unusual_location * 25) + (prepaid_card * 20)
    level = "HIGH" if score >= 50 else "MEDIUM" if score >= 25 else "LOW"
    return f"Risk Score: {score}/100 — {level} RISK"

sdk_tools = [sdk_get_order, sdk_get_customer_profile, sdk_retrieve_fraud_cases, sdk_calculate_risk_score]
print(f"✅ @function_tools registered: {[t.name for t in sdk_tools]}")


In [ ]:
# ── Agent definitions ──────────────────────────────────────────────────────────

# Phase 2: Evaluator Agent — no tools needed, pure reasoning + verdict
evaluator_agent = Agent(
    name="Senior Fraud Analyst",
    model="gpt-4o-mini",
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n\n"
        "You are a senior fraud analyst at an e-commerce company.\n"
        "You receive a junior AI agent's full investigation report.\n"
        "Use the risk score, historical case comparisons, and order details "
        "to issue your final independent verdict.\n\n"
        "Respond strictly in this format:\n"
        "DECISION: <APPROVE | FLAG FOR REVIEW | BLOCK>\n"
        "REASON: <one concise sentence citing specific evidence from the report>"
    ),
)

# Phase 1: Investigator Agent — calls tools, then hands off to evaluator
investigator_agent = Agent(
    name="Fraud Detection Agent",
    model="gpt-4o-mini",
    instructions=(
        f"{RECOMMENDED_PROMPT_PREFIX}\n\n"
        "You are a fraud detection agent for an e-commerce platform.\n"
        "For every order, follow these steps in order:\n"
        "1. Call sdk_get_order() to retrieve order details\n"
        "2. Call sdk_get_customer_profile() using the customer_id from the order\n"
        "3. Call sdk_retrieve_fraud_cases() with a short description of the key risk factors\n"
        "   — this returns 3 semantically similar historical cases from ChromaDB\n"
        "4. Call sdk_calculate_risk_score() with the appropriate boolean flags\n"
        "5. Write a structured assessment covering:\n"
        "   - Order & customer summary\n"
        "   - Risk score and what drove it\n"
        "   - What the similar historical cases suggest\n"
        "   - Your overall conclusion\n"
        "6. Hand off your complete assessment to the Senior Fraud Analyst for final verdict"
    ),
    tools=sdk_tools,
    # handoffs lets the agent autonomously transfer to the evaluator — no manual call needed
    handoffs=[handoff(evaluator_agent)],
)

print("✅ Investigator Agent and Evaluator Agent defined.")
print(f"   Investigator tools : {[t.name for t in investigator_agent.tools]}")
print(f"   Investigator handoffs: {[h.agent_name for h in investigator_agent.handoffs]}")


In [ ]:
SDK_ORDER = "ORD-1042"   # ← Try: ORD-1042, ORD-1043, ORD-1044, ORD-1045

print(f"{'═'*60}")
print(f"  OPENAI AGENTS SDK FRAUD SYSTEM  |  Order: {SDK_ORDER}")
print(f"{'═'*60}\n")
print("Runner.run_sync() — the SDK manages the loop, tool calls, and handoff\n")

# ── Single call replaces: fraud_agent_loop() + evaluate_assessment() ──────────
# The SDK handles:
#   • the while-True agent loop          (was fraud_agent_loop)
#   • detecting and executing tool calls (was fraud_handle_tool_calls)
#   • appending messages internally      (was messages.append / extend)
#   • triggering the evaluator handoff   (was the manual evaluate_assessment call)

result = Runner.run_sync(
    investigator_agent,
    input=f"Assess order {SDK_ORDER} for potential fraud.",
)

print(f"\n{'─'*60}")
print("FINAL VERDICT (from Senior Fraud Analyst via handoff)\n")
print(result.final_output)
print(f"\n{'═'*60}")

# ── Transparency: show every step the SDK took ────────────────────────────────
print("\n📋 SDK Run Items (full trace):\n")
for item in result.new_items:
    print(f"  [{type(item).__name__}] {getattr(item, 'agent', {})}")
